# Load and process Bacillus data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO, Phylo, AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load `.xmfa` data

In [2]:
def read_xmfa_biopython(file_path):
    blocks = list(AlignIO.parse(file_path, "mauve"))
    return blocks

In [3]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing a '-' completely removed.
    """
    align_len = alignment_block.get_alignment_length()
    
    valid_columns = [i for i in range(align_len) if "-" not in alignment_block[:, i]]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    return MultipleSeqAlignment(cleaned_records), valid_columns

In [4]:
xmfa_path = "../data/bacillus/bacillus.xmfa"
aligned_blocks = read_xmfa_biopython(xmfa_path)

print(f"Total blocks successfully loaded: {len(aligned_blocks)}")

Total blocks successfully loaded: 1218


In [5]:
def divide_evenly(n, parts):
    quotient, remainder = divmod(n, parts)
    
    # Create the list with the distributed remainder
    parts = [quotient + 1] * remainder + [quotient] * (parts - remainder)
    
    return parts

In [8]:
block_info_df = pd.read_csv("../data/bacillus/bacillus_block_info.csv")
block_info_df['Gene_Length_raw'] = None
block_info_df['Start_pos_raw'] = None
block_info_df['End_pos_raw'] = None
block_info_df.head()

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,0_0,5247,71581,76827,True,None,None,None
1,0_1,5246,76828,82073,True,None,None,None
2,1,4191,67350,71540,True,None,None,None
3,2,2175,28543,30717,True,None,None,None
4,3,2401,142577,144977,True,None,None,None


In [9]:
df_index = 0
for i in tqdm(range(len(aligned_blocks)), desc="Processing genes"):
    # get the raw block and remove gap columns
    raw_block = aligned_blocks[i]
    clear_block, clean_to_raw_map = remove_gap_columns(raw_block)

    if clear_block.get_alignment_length() <= 10000:
        start_pos_clean = block_info_df.loc[df_index, 'Start_pos']
        end_pos_clean = block_info_df.loc[df_index, 'End_pos']
        
        block_info_df.loc[df_index, 'Start_pos_raw'] = clean_to_raw_map[0] + start_pos_clean
        block_info_df.loc[df_index, 'End_pos_raw'] = clean_to_raw_map[-1] + start_pos_clean
        block_info_df.loc[df_index, 'Gene_Length_raw'] = block_info_df.loc[df_index, 'End_pos_raw'] - block_info_df.loc[df_index, 'Start_pos_raw'] + 1

        df_index += 1
    else:
        sub_segments = int(clear_block.get_alignment_length() / 10000) + 1
        sub_length = divide_evenly(clear_block.get_alignment_length(), sub_segments)
        sub_start = raw_block[0].annotations.get('start')

        for j in range(sub_segments):
            start_pos_clean = block_info_df.loc[df_index, 'Start_pos']
            end_pos_clean = block_info_df.loc[df_index, 'End_pos']

            block_info_df.loc[df_index, 'Start_pos_raw'] = clean_to_raw_map[start_pos_clean - sub_start] + sub_start
            block_info_df.loc[df_index, 'End_pos_raw'] = clean_to_raw_map[end_pos_clean - sub_start] + sub_start
            block_info_df.loc[df_index, 'Gene_Length_raw'] = block_info_df.loc[df_index, 'End_pos_raw'] - block_info_df.loc[df_index, 'Start_pos_raw'] + 1

            df_index += 1

Processing genes: 100%|██████████| 1218/1218 [00:55<00:00, 21.82it/s]


In [10]:
block_info_df

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,0_0,5247,71581,76827,True,5264,71581,76844
1,0_1,5246,76828,82073,True,5247,76845,82091
2,1,4191,67350,71540,True,4197,67350,71546
3,2,2175,28543,30717,True,2182,28543,30724
4,3,2401,142577,144977,True,2434,142577,145010
...,...,...,...,...,...,...,...,...
1272,1214,1363,5190561,5191923,True,1363,5190561,5191923
1273,1215_0,5051,5217306,5222356,True,5060,5217306,5222365
1274,1215_1,5050,5222357,5227406,True,5075,5222366,5227440
1275,1216,6764,5175949,5182712,True,6809,5175949,5182757


-----

In [21]:
print(block_info_df.head())
block_info_df.to_csv("../data/bacillus/bacillus_block_info_updated.csv", index=False)

  Gene_ID  Gene_Length  Start_pos  End_pos  Alignment Gene_Length_raw  \
0     0_0         5247      71581    76827       True            5264   
1     0_1         5246      76828    82073       True            5247   
2       1         4191      67350    71540       True            4197   
3       2         2175      28543    30717       True            2182   
4       3         2401     142577   144977       True            2434   

  Start_pos_raw End_pos_raw  
0         71581       76844  
1         76845       82091  
2         67350       71546  
3         28543       30724  
4        142577      145010  
